In [ ]:
#here are the packages I used to do this. Feel free to add to this as needed
import astropy.units as u
from astropy.io import (fits, ascii)
import numpy as np
from matplotlib import pyplot as plt
import scipy.integrate
from scipy.optimize import curve_fit
import pandas as pd

In [ ]:
#Constants
R_sun = 6.95700E8 #m
parsec = 3.0857E+16 #m

In [ ]:
#path
#set paths
phot_stars_path = r'../files/'
photometry_path = r'../files/'
temperature_path = r'../files/' #model grid
filter_path = r'../files/filter_response/' # filter response function path
model_path = r'../files/SED_Models/' #sed models

#READ CSV
phot_data = pd.read_csv(photometry_path + r'photometry.csv')
temperature_grid = pd.read_csv(temperature_path + r'vega_magnitude_models.csv')
star_data = pd.read_csv(phot_stars_path + r'photometric_datav4.csv')
print('--------------------------------------------------------------')
print(phot_data.columns.to_list())
print('-------------------------------------------------------------')
print(star_data.columns.to_list())
print('-------------------------------------------------------------')
print(temperature_grid.columns.to_list())
print('----------------------------stars------------------------------')
print(star_data['star'])
print('----------------------------filter------------------------------')
print(phot_data['Filter'])

star_list = list(np.array(star_data['star'], dtype = str))
filter_list = list(np.array(phot_data['Filter'], dtype = str))

In [ ]:
star = '29 Dra'
R_star = 5.7621 # this is HU Vir calculated radius
c = 2.9979245e8 # speed of light in m/s


In [ ]:
star_index = star_list.index(star) #use this to retrive info for star
Av = star_data['Av'][star_index] #Av
dist = star_data['distance'][star_index] #distance

cite for reddening: https://ui.adsabs.harvard.edu/scan/manifest/1989ApJ...345..245C

In [ ]:
#redening functions
a_IR = lambda x: 0.574 * x**1.61
b_IR = lambda x: -0.527 * x**1.61

def a_opt(x):
    y = x - 1.82
    a = (1 + 0.17699*y - 0.50447* (y**2) - 0.02427* (y**3)
         + 0.72085* (y**4) + 0.01979* (y**5) - 0.77530* (y**6) 
         + 0.32999* (y**7))
    return a
def b_opt(x):
    y = x - 1.82
    b = (1.41338*y + 2.28305*(y**2) + 1.07233*(y**3) 
         - 5.38434*(y**4) - 0.62251*(y**5) 
         + 5.30260*(y**6) - 2.09002*(y**7))
    return b

a_uv1 = lambda x: 1.752 - 0.316*x - 0.104/((x - 4.67)**2 + 0.341)
a_uv2 = lambda x: a_uv1(x) - 0.04473*(x - 5.9)**2 - 0.009779*(x - 5.9)**3

b_uv1 = lambda x: -3.090 + 1.825*x + 1.206/((x - 4.62)**2 + 0.263)
b_uv2 = lambda x: b_uv1(x) + 0.2130*(x - 5.9)**2 + 0.1207*(x - 5.9)**3

a_farUV = lambda x: -1.073 - 0.628*(x - 8) + 0.137*(x - 8)**2- 0.070*(x - 8)**3
b_farUV = lambda x: 13.670 + 4.257*(x - 8) - 0.420*(x - 8)**2 + 0.374*(x - 8)**3


def red(wav, Av):
    '''
    calculates at reddening at certain wavelength
    
    inputs:
    wav (float or float array) - wavelength in angstroms
    Av  (float) - the Av value
    
    outputs:
    A_lam (float or float array) - A_lam coeff 
    '''
    x = 1/(wav) * 1e-6/1e-10 #convert from angstrom to micro meters
    Rv = 3.1#Av/E_BV
    condlist = [((x >= 0.3) & (x < 1.1)), 
                ((x >= 1.1) & (x < 3.3)),
               ((x >= 3.3) & (x < 5.9)),
               ((x >= 5.9) & (x <= 8)),
               ((x >= 8) & (x <= 10))]
    functlistA = [a_IR, a_opt, a_uv1, a_uv2, a_farUV, 0]
    functlistB = [b_IR, b_opt, b_uv1, b_uv2, b_farUV,0]
    #piecewise functions
    a = np.piecewise(x, condlist, functlistA)
    b = np.piecewise(x, condlist, functlistB)
    
    A_lam = (a + b/Rv) * Av
    return A_lam

In [ ]:
modeling_data = phot_data[0:-1][['Filter', 'wav_eff', 'wav_mean']]

#name of filter files
filter_files = np.array([r'2MASS\2MASS_2MASS.J.dat', r'2MASS\2MASS_2MASS.H.dat', r'2MASS\2MASS_2MASS.Ks.dat',
               r'WISE\WISE_WISE.W1.dat', r'WISE\WISE_WISE.W2.dat',r'WISE\WISE_WISE.W3.dat',r'WISE\WISE_WISE.W4.dat',
               r'Gaia\GAIA_GAIA3.Gbp.dat', r'Gaia\GAIA_GAIA3.G.dat',r'Gaia\GAIA_GAIA3.Grp.dat',
               r'GALEX\GALEX_GALEX.FUV.dat', r'GALEX\GALEX_GALEX.NUV.dat'])

filter_names = np.array(modeling_data['Filter']) #filter names

In [ ]:
def star_flux(mag, zero_point, m0=0):
    flux = 10**(-(mag - m0)/2.5) * zero_point
    return flux


def star_photometry(filt_name, return_err = False):
    '''
    Gather fluxes (and errors) from star photometry data
    inputs:
        filt_name (str) name of filter
        return_err (bool) - return error. For this code to work set return_err to false
    outputs
    - flux (float) - flux of star in certain filter
    - flux_error (float) - flux error. Only returns if return_err = True
    
    '''
    #get filter
    filt_index = filter_list.index(filt_name)
    #gather ZP
    ZP = float(phot_data['ZP (Vega)'][filt_index])
    
    #GALEX Filters
    if (filt_name == 'GALEX FUV') or (filt_name == 'GALEX NUV'):
        #get set m0, zeropoint (DO NOT CHANGE)
        if filt_name == 'GALEX FUV':
            cts_factor = 1.40e-15
            m0 = 18.82
        else:  #nuv mag, zeropoint
            cts_factor = 2.06e-16
            m0 = 20.08
        
        #get magnitude, error
        mag_ab = star_data[filt_name][star_index] 
        
        #get  magnitude error
        filt_name2 = filt_name + ' Error'
        mag_err = star_data[filt_name2][star_index]
        
        #calculate  flux
        flux = star_flux(mag_ab, cts_factor, m0) #flux
        #calculate flux error
        flux_err = np.abs(star_flux(np.array([mag_ab - mag_err, mag_ab + mag_err]),
                                    cts_factor, m0) - np.array([flux,flux]))
    #GAIA FLUXES        
    else: 
        if (filt_name == 'Gaia BP') or (
            filt_name == 'Gaia G') or (filt_name == 'Gaia RP') :
            # Gather magnitudes, flux_errors
                #magnitudes  
                filt_name2 = 'm_' + filt_name[5:len(filt_name)]
                mag = star_data[filt_name2][star_index]
                #flux errors
                filt_name2 = filt_name + ' flux_over_error' 
                flux_err = star_data[filt_name2][star_index]
                #calculate flux
                flux = star_flux(mag, ZP, 0)
                #calculate error
                flux_err = np.array([1/flux_err * flux, 1/flux_err * flux])
        #ALLWISE + 2MASS Fluxes      
        else:
            #fluxes
            mag = star_data[filt_name][star_index]
            flux = star_flux(mag, ZP, 0)
            #errors
            filt_name2 = filt_name + ' error'
            mag_err = star_data[filt_name2][star_index]
            flux_err = np.abs(star_flux(np.array([mag - mag_err, mag + mag_err]),
                                    ZP, 0) - np.array([flux,flux]))
    
    if return_err == True:
        return [flux, flux_err[0], flux_err[1]]
    else:
        return flux #only returns fluxes

def model_fluxes(model_file, logg):
    '''
    filter_file (str) - path of the filter transmission file
    model_file (str) - path of ATLAS9 model file
    logg (str) - surface gravity
    '''
    
    model_data = fits.open(model_file)[1].data
    wav = model_data['WAVELENGTH']
    reddening = 10**-(red(wav, Av)/2.5)
    model_flux = model_data[logg]
    return wav, model_flux, reddening
    

def filter_transmission(filter_file, wav, model_flux, return_original = False):
    filter_file = filter_file
    filt_data = np.genfromtxt(filter_file, dtype= float)
    filt_wav = np.array(filt_data[:,0], dtype = float)
    filt_flux = np.array(filt_data[:,1], dtype = float)
    
    print(len(filt_flux) == len(filt_wav))
    
    if return_original == True:
        return filt_wav, filt_flux
    else:
        filt_interflux = np.interp(wav, filt_wav, filt_flux) 
        return filt_interflux
power_law = lambda x, a, k: a*np.power(x,-k)    
    
def model_filt_flux(filt_name, filter_file, wav, flux, inter_wav = -1, inter_flux = -1, rad = 1, interred = -1):
    filt_index = filter_list.index(filt_name)
    #gather wav_eff, Weff
    cent_wav = float(phot_data['wav_eff'][filt_index])
    W_eff = float(phot_data['Weff'][filt_index])
    
    #check for interpolated function
    if (type(inter_wav) == int) or (type(inter_flux)  == int):
        inter_wav = wav
        inter_flux = flux
        
    #gather filter transmission data
    filter_file =  filter_file
    filt_data = np.genfromtxt(filter_file, dtype= float)
    filt_wav = np.array(filt_data[:,0], dtype = float)
    filt_flux = np.array(filt_data[:,1], dtype = float)
        
    #Normalize
    div = 1
    if max(filt_flux) > 1:
        div = max(filt_flux)
    
    #artifically set ends to 0
    if min(filt_flux) > 0:
        filt_flux[-1] =   0  #array, where to put, new item
        filt_flux[0] =   0 
    
    #interpolate flux
    filt_interflux = np.interp(inter_wav, filt_wav, filt_flux)/div
    
    #calculate flux
    filt = np.where((inter_wav >= cent_wav - rad) & (inter_wav <= cent_wav + rad))
    av_flux = scipy.integrate.trapezoid(filt_interflux, x = inter_wav)/W_eff #find average flux
    
    #calculate redening
    if type(interred) == int:
        filter_red = 1
    else:
        #interred = np.interp(inter_wav, wav, red)
        filter_red = np.median(interred[filt])
    return av_flux * np.median(inter_flux[filt]) * filter_red


def get_flux_models(wavelength, model_flux, red, lim = 10**4.8):
    '''
    get flux models for a given SED 
    
    inputs:
    - wavelength (float array) - wavelength of SED MODEL
    - model_flux (float array) - SED MODEL
    - red (float array) - reddening
    '''
    #interpolate
    int_wav = np.arange(min(wavelength), max(wavelength), 1)
    int_model_flux = np.interp(int_wav, wavelength, model_flux)
    int_red = np.interp(int_wav, wavelength, red)
    
    #curve fit 
    (a1, k1), pcov = curve_fit(power_law, wavelength[wavelength > lim], model_flux[wavelength > lim], p0=[10e12, 3.42])
    int_model_flux[int_wav > lim]  = power_law(int_wav[int_wav > lim], a1, k1)
    

    #zip filter_files, with filter names
    temp_list = np.array(list(zip(filter_files, filter_names)))
    
    #get model fluxes
    flux_list = np.array([model_filt_flux(item[1], filter_path + item[0], wav = wavelength, flux = model_flux,
                                              inter_wav = int_wav, inter_flux = int_model_flux, 
                                               interred = int_red) for item in temp_list])
    return flux_list
        

In [ ]:
#calculate j-h color
Ebp_rp = Av*0.789/1.890 
j_mass = star_data['2MASS J'][star_index] - 0.243 *Av
h_mass = star_data['2MASS H'][star_index] - 0.131 *Av
j_h =  j_mass - h_mass
print('J-H', j_h)
temperature_grid[['T_eff', 'J-H']]

In [ ]:
logg = 4 #set logg
i_logg = 'g' + str(int(logg*10)) 

#SET Temperature
temperature1 = 4500
temperature2 = 4750
temperature3 = 5000



#get wavelength, sed model, reddening
wavelength, model_flux1, reddening = model_fluxes(model_path + f'ckp00\ckp00_{temperature1}.fits', i_logg)
wavelength, model_flux2, reddening = model_fluxes(model_path + f'ckp00\ckp00_{temperature2}.fits', i_logg)
wavelength, model_flux3, reddening = model_fluxes(model_path + f'ckp00\ckp00_{temperature3}.fits', i_logg)


#get observed fluxes, filter wavelengths
obs_fluxes = np.array([star_photometry(item, return_err = False) for item in filter_names])
λeffs = np.array(modeling_data['wav_eff'])

In [ ]:
#get model fluxes
mod_fluxes_1 = get_flux_models(wavelength, model_flux1, reddening)
print(1)
mod_fluxes_2 = get_flux_models(wavelength, model_flux2, reddening)
print(2)
mod_fluxes_3 = get_flux_models(wavelength, model_flux3, reddening)
print(3)


First, lets read in a stellar model of an A0 main sequence star. A fits file containing the model was provided with the lab ('ckm05_9500.fits'). This is a Castelli-Kurucz stellar model of an A0 main sequence star with [Fe/H]= -0.5. More information on the models can be found here: https://archive.stsci.edu/hlsps/reference-atlases/cdbs/grid/ck04models/AA_README

You should see that the file contains several different columns that correspond to a difference surface gravity (log g, where g is the gravitational acceleration at the surface of the star). We want to use log(g)= 4.0, so we will want to use the "g40" column for the analysis that follows.

Plot of this model SED (flux on the y axis, wavelength on the x axis). Scale both axes logarithmically. Include axis labels and don't forget units. You can find information on units in the table header and the documentation about Castelli-Kurucz models.

## Plot SED MODELS with Multiple Temperatures

In [ ]:
ylabel_orient = ['bottom', 'top', 'top',
                 'top', 'top', 'top', 'top',
                 'top', 'bottom', 'bottom', 'top', 'top']
xlabel_orient = ['right', 'right', 'left',
                 'left', 'left', 'left', 'left',
                 'right', 'right', 'right', 'left', 'left']

#Calculate Radius from vsini
adjust_radius = True # if star has vsini use the minimum radius set to False
vsini = star_data['vsini'][star_index]*1e3
P_rot = star_data['VSX Period'][star_index]
ω_rot = 2*np.pi/(P_rot * 24 *3600) #for consistency sake 
R_min = vsini/ω_rot / R_sun

if (R_min > -999) and (adjust_radius == False): #automatically set R_star to R_min
    R_star = R_min
else:
    R_star = 3

star_label = star.split(' ')[0] + '_' + star.split(' ')[1]
    
flux_scale = ((R_star * R_sun)/ (dist * parsec))**2 #flux scale

fig, ax = plt.subplots(1, figsize = (7, 5)) #

#SED MODELS
ax.set_title(f'{star} SED, R = {R_star} ' + r'$R_{\odot}$')
ax.plot(wavelength, model_flux1 * flux_scale* reddening, 
        color = 'black', label = f'SED Model (T = {temperature1} K, logg = {logg})')
ax.plot(wavelength, model_flux2 * flux_scale * reddening, 
        color = 'slategrey', label = f'SED Model (T = {temperature2} K, logg = {logg})' )
ax.plot(wavelength, model_flux3 * flux_scale * reddening, 
        color = 'grey', label = f'SED Model (T = {temperature3} K, logg = {logg} )')

ax.errorbar(λeffs, obs_fluxes , yerr = [obs_fluxes*0.1315, obs_fluxes*0.1315], 
            fmt = 'none', color = 'tab:cyan', capsize=3, lw = 3)

#Flux MODELS
ax.plot(λeffs, mod_fluxes_3 * flux_scale, ls = 'none', marker = 'o', 
        ms = '5', color = 'green', label = f'Model Flux (T = {temperature3} K)')

ax.plot(λeffs, mod_fluxes_2 * flux_scale, ls = 'none', marker = '^', 
        ms = '5', color = 'orange', label = f'Model Flux (T = {temperature2} K)')

ax.plot(λeffs, mod_fluxes_1 *flux_scale, ls = 'none', marker = 'P', 
        ms = '5', color = 'red', label = f'Model Flux (T = {temperature1} K)')
#Observed Fluxes
ax.plot(λeffs, obs_fluxes, ls = 'none', marker = '*', 
        ms = '5', color = 'blue', label = 'Observed Flux')

#Label Filters
for i in range(len(λeffs)):
    ax.text(λeffs[i], mod_fluxes_1[i] * flux_scale, filter_names[i], va = ylabel_orient[i])

    
    
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Flux $(erg/cm^{2}/s/A)$')
ax.set_yscale('log')
ax.set_xscale('log')

ax.legend(fontsize=9)

ax.set_xlim(10**3, 10**-5.5)


#adjust ylims
coeff_high = np.ceil(np.log10(mod_fluxes_2[8] * flux_scale))+0.5
coeff_low = np.floor(np.log10(mod_fluxes_2[10] * flux_scale))-0.5


high_ylim = 10**coeff_high
low_ylim = 10**coeff_low


#adjust ylims
ax.set_xlim(10**3, 10**5.5)
ax.set_ylim(low_ylim, high_ylim)
#ax.set_ylim(10e-20, 10e-14)

In [ ]:
# print temperatures as a reminder
print(f'T1: {temperature1}')
print(f'T2: {temperature2}')
print(f'T3: {temperature3}')

## Final SED MODEL

In [ ]:
filter_label = ['$J$', '$H$', '$K_{s}$', '$W1$', '$W2$', '$W3$', '$W4$',
 '$G_{BP}$', '$G$', '$G_{RP}$', '$FUV$', '$NUV$']
final_yorient = ['bottom', 'bottom', 'bottom',
                 'bottom', 'bottom', 'bottom', 'bottom',
                 'top', 'top', 'top', 'bottom', 'bottom']
xlabel_orient = ['right', 'right', 'left',
                 'left', 'left', 'left', 'left',
                 'right', 'right', 'right', 'left', 'left']



#pick correct model according to temperature
#for example temperature1 --> model_flux1 (SED MODEL), mod_fluxes1 (model fluxes)
model_flux = model_flux2
final_temperature = temperature2
mod_fluxes = mod_fluxes_2

fig, ax = plt.subplots(1, figsize = (6, 5))

#Models
ax.set_title(f'{star} SED, R = {R_star} ' + r'$R_{\odot}$')
ax.plot(wavelength, model_flux * flux_scale* reddening, 
        color = 'black',
        label = f'SED Model (T = {final_temperature} K, logg = {logg})')



ax.plot(λeffs, mod_fluxes *flux_scale , ls = 'none', marker = 'o', 
        ms = '5', color = 'green', label = f'Model Flux (T = {final_temperature} K)')


ax.errorbar(λeffs, obs_fluxes , yerr = [obs_fluxes * 0.3, obs_fluxes * 0.3], 
            fmt = 'none', color = 'tab:cyan', capsize=3, lw = 3)
ax.plot(λeffs, obs_fluxes, ls = 'none', marker = '*', 
        ms = '5', color = 'blue', label = 'Observed Flux')

for i in range(len(λeffs)):
    ax.text(λeffs[i], mod_fluxes[i] * flux_scale, filter_label[i], va = final_yorient[i])

    
ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Flux $(erg/cm^{2}/s/A)$')
ax.set_yscale('log')
ax.set_xscale('log')

ax.legend(fontsize=9)

#adjust ylims
max_flux = mod_fluxes[8] * flux_scale
if obs_fluxes[-2] >  max_flux:
    max_flux = obs_fluxes[-2]
    print(obs_fluxes[-2])
    
coeff_low = np.ceil(np.log10(max_flux))
coeff_high = np.floor(np.log10(mod_fluxes[6] * flux_scale))-0.5

print(coeff_low)
low_ylim = 10**coeff_low
high_ylim = 10**coeff_high


#adjust ylims
ax.set_xlim(10**3, 10**5.5)
ax.set_ylim(high_ylim, 10**-13.5)#high_ylim, low_ylim) #10e-20, 10e-11)

#change image path to desired path
image_path = r'SED_images/uv_excess_seds//ssg_seds/' + star_label + '_SED.png'
fig.savefig(image_path, dpi = 300)

In [ ]:
star_data[['star','2MASS_qph','ALLWISE_qph']][star_index:star_index+1]

In [ ]:
print(flux_scale)